In [208]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelBinarizer,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score



In [209]:
import os
print(os.getcwd())
print(os.listdir())


C:\Users\Bryon\PycharmProjects\PythonProject\uwgb-comp-sci-464-group-project
['.git', '.gitignore', '.idea', '.venv', 'CODEOWNERS', 'data_sources', 'docs', 'examples', 'feature_Importance.ipynb', 'Learning_Exercies', 'README.md', 'requirements.txt']


In [210]:
base_dir = "./data_sources/prosafe_dataset"
policy_from_2011_to_2014 = pd.read_csv(f"{base_dir}/policy_from_2011_to_2014.csv")
policy_from_2014_to_2018 = pd.read_csv(f"{base_dir}/policy_from_2014_to_2018.csv")

In [211]:
# Step 1: Choose Input Feature(s) and Corresponding Predicted Output Feature

target_col = "PREMIUM"

y = policy_from_2011_to_2014['PREMIUM']
X = policy_from_2011_to_2014.drop(columns=['PREMIUM'])

In [212]:
# 0. DROP features with >50% missing values (and CCM_TON you decided to drop)
X = X.drop(columns=['CCM_TON', 'CARRYING_CAPACITY', 'CLAIM_PAID'])

# Drop rows where target PREMIUM (y) is NaN
mask = y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].copy()

# Fix EFFECTIVE_YR
X['EFFECTIVE_YR'] = pd.to_numeric(X['EFFECTIVE_YR'], errors='coerce')
X['EFFECTIVE_YR'] = X['EFFECTIVE_YR'].fillna(X['EFFECTIVE_YR'].median())

# Fix a few other numeric columns with small NaN counts
X['SEATS_NUM'] = X['SEATS_NUM'].fillna(X['SEATS_NUM'].median())
X['PROD_YEAR'] = X['PROD_YEAR'].fillna(X['PROD_YEAR'].median())

# Date columns → ordinals
date_cols = ["INSR_BEGIN", "INSR_END"]
for col in date_cols:
    X[col] = pd.to_datetime(X[col], format='%d-%b-%y', errors='coerce')
    X[col] = X[col].map(lambda x: x.toordinal() if pd.notnull(x) else np.nan)

# Fill any remaining numeric NaNs (e.g., from bad dates)
X = X.fillna(X.median(numeric_only=True))

# One-hot encode categoricals
cat_cols = ['TYPE_VEHICLE', 'MAKE', 'USAGE']
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Final sanity check


# 2. Split AFTER preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (234823, 487)
X_test shape: (58706, 487)


In [213]:
# 1. See which columns are still object (string) type
obj_cols = X_train.select_dtypes(include=['object']).columns
print("Object columns:", list(obj_cols))


Object columns: []


In [214]:
# TRAIN

rf = RandomForestRegressor(
    n_estimators=50,    # way fewer trees than 300
    max_depth=15,       # cap tree depth
    max_features='sqrt',# try fewer features per split
    n_jobs=-1,          # all cores
    random_state=42
)

rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
print("R²:", r2_score(y_test, y_pred))



R²: 0.6567755199362557
